#### 1. Data import and device

In [ ]:
from IPython.display import display
import os
import torch
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as T
from pathlib import Path
from collections import Counter
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score
import numpy as np
import torch.nn as nn
import gc
from matplotlib import pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from PIL import Image,UnidentifiedImageError
import imagehash
from tqdm import tqdm
from collections import defaultdict

In [ ]:
# --- config ---
ROOT = Path().resolve()
DATA_DIR = os.path.join(ROOT, "data", "archive", "train")
TEST_DIR = os.path.join(ROOT, "data", "archive", "test")
IMG_SIZE = 224
BATCH_SIZE = 32
VAL_SPLIT = 0.2
NUM_WORKERS = 4
PIN = torch.cuda.is_available()
device = torch.device("cuda" if PIN else "cpu")
torch.multiprocessing.set_sharing_strategy('file_system')

# 设置normalize参数
IMAGENET_MEAN = (0.5, 0.5, 0.5)
IMAGENET_STD = (0.5, 0.5, 0.5)

#对训练图片进行各种操作和加强,包括颜色增强,翻转,旋转
train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(0.5),
    T.RandomRotation(10),
    T.ColorJitter(0.1, 0.1, 0.05),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

#对验证集做部分操作
val_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

#对测试集做部分操作
test_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def get_loaders(
        data_dir: str = DATA_DIR,
        batch_size: int = BATCH_SIZE,
        val_split: float = VAL_SPLIT,
        num_workers: int = NUM_WORKERS,
        pin_memory: bool = PIN,
):
    full = torchvision.datasets.ImageFolder(root=data_dir, transform=train_tf)
    n = len(full)
    n_val = int(n * val_split)
    n_train = n - n_val

    train_set, val_set = random_split(full, [n_train, n_val])

    # clean tf for val
    val_set.dataset = torchvision.datasets.ImageFolder(root=data_dir, transform=val_tf)

    class_to_idx = full.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory, drop_last=False
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size * 2, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory, drop_last=False
    )
    return train_loader, val_loader, idx_to_class

test_dataset = torchvision.datasets.ImageFolder(
    root=TEST_DIR,   # 改成你的 test 路径
    transform=test_tf
)

def get_test_set():
    test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
    )
    return test_loader

train_loader, val_loader, idx_to_class = get_loaders()

xb, yb = next(iter(train_loader))

#### 2.Data visualization

show image amount, show class amount,and batch size

In [ ]:
# ===== Dataset Overview =====

dataset = torchvision.datasets.ImageFolder(root=DATA_DIR)
class_counts = Counter(dataset.targets)

class_table = pd.DataFrame(
    {
        "Class Index": list(range(len(dataset.classes))),
        "Class Name": dataset.classes,
        "Image Count": [class_counts[i] for i in range(len(dataset.classes))],
    }
)

overview = pd.DataFrame(
    {
        "Metric": [
            "Total number of images",
            "Number of classes",
            "Training samples",
            "Validation samples",
            "Train batch size",
            "Validation batch size",
            "Image size",
        ],
        "Value": [
            len(dataset),
            len(dataset.classes),
            len(train_loader.dataset),
            len(val_loader.dataset),
            train_loader.batch_size,
            val_loader.batch_size,
            f"{IMG_SIZE}x{IMG_SIZE}",
        ],
    }
)

display(overview.style.hide(axis="index"))

Display the first 10 images from the dataset.

In [ ]:
# Compare original images with augmented images from train_tf

def denormalize_image(image_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    mean_tensor = torch.tensor(mean).view(3, 1, 1)
    std_tensor = torch.tensor(std).view(3, 1, 1)
    image_tensor = image_tensor.detach().cpu() * std_tensor + mean_tensor
    return image_tensor.clamp(0, 1).permute(1, 2, 0).numpy()

num_compare = 10
fig, axes = plt.subplots(num_compare, 2, figsize=(10, 4 * num_compare))

for i in range(num_compare):
    original_img, label = dataset[i]
    augmented_img = train_tf(original_img)

    axes[i, 0].imshow(original_img)
    axes[i, 0].set_title(f"Original\n{dataset.classes[label]}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(denormalize_image(augmented_img))
    axes[i, 1].set_title(f"Augmented\n{dataset.classes[label]}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

Count images per class to check class balance.

In [ ]:
plot_data = class_table.assign(
    **{"Class Index": class_table["Class Index"].astype(str)}
)

fig, (ax_table, ax_bar) = plt.subplots(
    1,
    2,
    figsize=(22, max(10, len(class_table) * 0.4)),
    gridspec_kw={"width_ratios": [2.0, 2.2]},
)

ax_table.axis("off")
table = ax_table.table(
    cellText=class_table[["Class Index", "Class Name"]].values,
    colLabels=["Class Index", "Class Name"],
    loc="center",
    cellLoc="left",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)
ax_table.set_title("Class Table", fontsize=14, pad=12)

sns.barplot(
    data=plot_data,
    x="Class Index",
    y="Image Count",
    hue="Class Index",
    dodge=False,
    palette="viridis",
    legend=False,
    ax=ax_bar,
  )
ax_bar.set_title("Images Per Class", fontsize=14)
ax_bar.set_xlabel("Class Index")
ax_bar.set_ylabel("Image Count")
ax_bar.tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

#### 3. Data cleaning

In [ ]:
### 3. Data Cleaning (Robust & Multi-Threaded English Version)


# ─────────────────────────────────────────────────────────────────────
# Step 1: Corrupted & Tiny Image Detection
# ─────────────────────────────────────────────────────────────────────
def verify_image(file_path: Path) -> Path | None:
    """Checks if an image is corrupted or too small to be useful."""
    try:
        with Image.open(file_path) as img:
            img.verify()

        with Image.open(file_path) as img:
            if img.size[0] < 10 or img.size[1] < 10:
                return file_path
        return None
    except (IOError, SyntaxError, UnidentifiedImageError, Exception):
        return file_path

print("=" * 60)
print("Phase 1: Scanning for Corrupted or Invalid Images...")
print("=" * 60)

# Gather all image paths
image_extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
all_images = [p for p in Path(DATA_DIR).rglob('*') if p.suffix.lower() in image_extensions]
original_count = len(all_images)
print(f"Total images found: {original_count}\n")

# Parallel Verification
with ThreadPoolExecutor(max_workers=NUM_WORKERS * 2) as executor:
    results = list(tqdm(executor.map(verify_image, all_images),
                        total=len(all_images), desc="Verifying Integrity"))

corrupted_files = [res for res in results if res is not None]
# Filter out corrupted files for the next step
valid_images_list = [p for p in all_images if p not in set(corrupted_files)]

# ─────────────────────────────────────────────────────────────────────
# Step 2: Perceptual Hashing — Duplicates & Label Conflicts
# ─────────────────────────────────────────────────────────────────────
def compute_phash(file_path: Path) -> tuple[Path, str | None]:
    """Computes the Perceptual Hash (pHash) of an image."""
    try:
        with Image.open(file_path) as img:
            return file_path, str(imagehash.phash(img))
    except Exception:
        return file_path, None

print("\n" + "=" * 60)
print("Phase 2: Running pHash — Detecting Duplicates & Label Conflicts...")
print("=" * 60)

with ThreadPoolExecutor(max_workers=NUM_WORKERS * 2) as executor:
    hash_results = list(tqdm(executor.map(compute_phash, valid_images_list),
                             total=len(valid_images_list), desc="Computing Hashes"))

# Map hashes to paths and labels: {hash: [(path, label), ...]}
hash_map = defaultdict(list)
for file_path, h in hash_results:
    if h is not None:
        label = file_path.parent.name
        hash_map[h].append((file_path, label))

# Analyze duplicates and conflicts
duplicates_to_remove = set()  # Same image, same label
conflict_files = set()       # Same image, DIFFERENT labels

for h, entries in hash_map.items():
    if len(entries) > 1:
        unique_labels = {label for _, label in entries}

        if len(unique_labels) > 1:
            # MEDICAL SAFETY STRATEGY:
            # If an image has conflicting labels, discard the entire group.
            for p, _ in entries:
                conflict_files.add(p)
        else:
            # Simple Duplicates: Keep the first one, flag the rest for removal.
            for p, _ in entries[1:]:
                duplicates_to_remove.add(p)

# ─────────────────────────────────────────────────────────────────────
# Step 3: Summary Report & Physical Cleanup
# ─────────────────────────────────────────────────────────────────────
files_to_remove = set(corrupted_files) | duplicates_to_remove | conflict_files

# Print conflict examples for debugging
if conflict_files:
    print(f"\nAlert: Label Conflicts Detected! Identical images found in multiple categories:")
    conflict_count = 0
    for h, entries in hash_map.items():
        labels = {label for _, label in entries}
        if len(labels) > 1:
            print(f"   Hash [{h}] → Shared by labels: {labels}")
            conflict_count += 1
            if conflict_count >= 3: break

# Execute physical deletion
for p in files_to_remove:
    if p.exists():
        p.unlink()

final_count = original_count - len(files_to_remove)

print("\n" + "=" * 60)
print("                  DATA CLEANING SUMMARY")
print("=" * 60)
print(f"  {'Initial Image Count:':<35} {original_count:>6}")
print(f"  {'Corrupted / Invalid (Deleted):':<35} {len(corrupted_files):>6}")
print(f"  {'Duplicate Images (Deleted):':<35} {len(duplicates_to_remove):>6}")
print(f"  {'Label Conflicts (Entirely Removed):':<35} {len(conflict_files):>6}")
print("-" * 60)
print(f"  {'Total Images Removed:':<35} {len(files_to_remove):>6}")
print(f"  {'Final Cleaned Dataset Size:':<35} {final_count:>6}")
print(f"  {'Data Retention Rate:':<35} {(final_count/original_count)*100:>5.2f}%")
print("=" * 60)

#### 4. Model and comparison model

Our own 20-layer CNN Model

In [ ]:
class CNN20Model(nn.Module):
    def __init__(self, num_classes=23):
        super(CNN20Model, self).__init__()

        # Convolutional Layers
        self.features = nn.Sequential(
            # Block 1 (Input: 3x224x224 -> Output: 64x112x112)
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2 (Output: 128x56x56)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3 (Output: 256x28x28)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 4 (Output: 512x14x14)
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 5 (Output: 512x7x7)
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Classifier (Fully Connected Layers)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(4096, 1000),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(1000, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

#### 5. Model training

In [ ]:
#整个训练集被模型完整学习 30次
EPOCHS = 30
#每次更新参数的“步子大小”
LEARNING_RATE = 1e-3
# 对模型参数加“惩罚”（L2正则）,控制模型复杂度,提高泛化能力
WEIGHT_DECAY = 1e-4
#早停 防止过拟合
PATIENCE = 5

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

gc.collect()

def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    """Validate the model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE,
                weight_decay=WEIGHT_DECAY, patience=PATIENCE, model_name="model"):
    """Complete training loop with early stopping and checkpointing"""
    model = model.to(device)
    subset = train_loader.dataset
    targets = [subset.dataset.targets[i] for i in subset.indices]

    counts = Counter(targets)
    num_classes = len(counts)
    total_samples = sum(counts.values())

    class_weights = torch.tensor([
        total_samples / counts[i] for i in range(num_classes)
    ], dtype=torch.float).to(device)
    # 计算loss根据图片数量进行相应惩罚
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    best_val_acc = 0.0
    best_epoch = 0
    epochs_no_improve = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    print(f"\nStarting training for {model_name}...")
    print("-" * 70)

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        scheduler.step(val_loss)

        print(f"Epoch {epoch + 1:2d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:6.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:6.2f}%", end="", flush=True)

        # Early stopping & checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch + 1
            epochs_no_improve = 0
            torch.save(model.state_dict(), f'best_{model_name}.pth')
            print(f" ✓ [Best: {val_acc:.2f}%]")
        else:
            epochs_no_improve += 1
            print(flush=True)

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch + 1}")
            print(f"Best validation accuracy: {best_val_acc:.2f}% (epoch {best_epoch})")
            break

    # Load best weights
    model.load_state_dict(torch.load(f'best_{model_name}.pth'))
    print(f"\nTraining complete! Best model saved as 'best_{model_name}.pth'")
    print("-" * 70)
    return model, history


# Train Custom Model
print("\n" + "=" * 70)
print("TRAINING CUSTOM 20-LAYER CNN MODEL")
print("=" * 70)
NUM_CLASSES = len(idx_to_class)
print(f"Initializing 20-layer CNN for {NUM_CLASSES} classes on device: {device}")
model_20 = CNN20Model(num_classes=NUM_CLASSES).to(device)
model_20, model_20_history = train_model(
    model_20, train_loader, val_loader,
    model_name="cnn_20"
)

#### 6. Model performance comparison

In [ ]:
def plot_training_histories( history, model_name):
    """Compare training histories of three models side by side"""
    fig = plt.figure(figsize=(20, 12))

    epochs1 = range(1, len(history['train_loss']) + 1)

    # Model 1 - Loss
    ax1 = fig.add_subplot()
    ax1.plot(epochs1, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax1.plot(epochs1, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    ax1.set_title(f'{model_name} - Loss', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)



    # Model 1 - Accuracy
    ax4 = fig.add_subplot()
    ax4.plot(epochs1, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
    ax4.plot(epochs1, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
    ax4.set_title(f'{model_name} - Accuracy', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Accuracy (%)', fontsize=12)
    ax4.set_xlabel('Epoch', fontsize=12)
    ax4.legend(fontsize=11)
    ax4.grid(True, alpha=0.3)




# Plot comparison of all three models
plot_training_histories(model_20_history, 'Custom CNN-20')


In [ ]:
def evaluate_model_pytorch(model, loader, device, k=3):
    """Evaluate PyTorch model and return metrics"""
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    total_loss = 0.0
    correct = 0
    total = 0

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    avg_loss = total_loss / total
    accuracy = 100. * correct / total

    # Calculate top-k accuracy
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    top_k_acc = top_k_accuracy_score(all_labels, all_probs, k=k) * 100

    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'top_k_accuracy': top_k_acc,
        'predictions': np.array(all_preds),
        'probabilities': all_probs,
        'labels': all_labels
    }


print("Evaluating Custom CNN-20...")
model_20_results = evaluate_model_pytorch(model_20, val_loader, device, k=3)

# Create comparison table
results_df = pd.DataFrame({
    'Model': ['Custom CNN-20'],
    'Validation Loss': [ model_20_results['loss']],
    'Validation Accuracy (%)': [model_20_results['accuracy']],
    'Top-3 Accuracy (%)': [model_20_results['top_k_accuracy']]
})

print("\n" + "=" * 70)
print("=" * 70)
display(results_df.style.format({
    'Validation Loss': '{:.4f}',
    'Validation Accuracy (%)': '{:.2f}',
    'Top-3 Accuracy (%)': '{:.2f}'
}).hide(axis="index"))

In [ ]:
class_names = list(idx_to_class.values())
print("\n" + "=" * 70)
print("CUSTOM CNN-20 CLASSIFICATION REPORT")
print("=" * 70)
print(classification_report(model_20_results['labels'], model_20_results['predictions'],
                          target_names=class_names, digits=3))

def plot_confusion_matrix(y_true, y_pred, model_name, class_names):
    """Plot confusion matrix heatmap"""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(18, 15))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} - Confusion Matrix', fontsize=20, fontweight='bold')
    plt.ylabel('True Label', fontsize=15)
    plt.xlabel('Predicted Label', fontsize=15)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(model_20_results['labels'], model_20_results['predictions'],
                     'Custom CNN-20', class_names)


In [ ]:
test_loader = get_test_set()
criterion = nn.CrossEntropyLoss()

test_loader = get_test_set()

test_loss, test_acc = validate(model_20, test_loader, criterion, device)

model_20_test_results = evaluate_model_pytorch(model_20, test_loader, device, k=3)

results_df = pd.DataFrame({
    'Model': ['Custom CNN-20'],
    'Test Loss': [model_20_test_results['loss']],
    'Test Accuracy (%)': [model_20_test_results['accuracy']],
    'Top-3 Test Accuracy (%)': [model_20_test_results['top_k_accuracy']]
})

print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)
display(results_df.style.format({
    'Test Loss': '{:.4f}',
    'Test Accuracy (%)': '{:.2f}',
    'Top-3 Test Accuracy (%)': '{:.2f}'
}).hide(axis="index"))



plot_confusion_matrix(model_20_test_results['labels'], model_20_test_results['predictions'],'Custom CNN-20', class_names)